# Trimind v1 — Fine-tune codys12/Qwen3-8B-BitNet

Fine-tunes one openaiformat JSONL file at a time using LoRA.
Each file fits in a single Colab session (~30-60 min on T4).
Adapters saved locally and optionally pushed to Hugging Face Hub.

In [ ]:
# @title 1. Clone repo and install deps
import os, sys

REPO_URL = "https://github.com/Abdulhadi446/trimind-bitnet.git"
BRANCH = "main"

if not os.path.exists("/content/trimind-bitnet"):
    !git clone -b {BRANCH} {REPO_URL} /content/trimind-bitnet
else:
    %cd /content/trimind-bitnet
    !git pull

%cd /content/trimind-bitnet
!pip install -q -r training/requirements-colab.txt
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
# @title 2. Hardware check
import torch, psutil, subprocess

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
mem = psutil.virtual_memory()
print(f"RAM: {mem.total / 1e9:.1f} GB ({mem.available / 1e9:.1f} GB free)")
result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
print(f"Disk:\n{result.stdout}")

In [ ]:
# @title 3. Login to Hugging Face
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("Set HF_TOKEN in Colab secrets (key icon in sidebar)")

In [ ]:
# @title 4. Upload dataset cache from Google Drive (optional)
from google.colab import drive
import os

DRIVE_CACHE = "/content/drive/MyDrive/trimind_data"
LOCAL_DATA = "/content/trimind-bitnet/data"

if os.path.exists(DRIVE_CACHE) and not os.path.exists(LOCAL_DATA):
    !ln -s "{DRIVE_CACHE}" "{LOCAL_DATA}"
    print("Linked Drive cache to ./data")
elif not os.path.exists(LOCAL_DATA):
    print("No cache found. Data will be downloaded fresh.")
else:
    print("Data dir already exists.")

In [ ]:
# @title 5. Test run (50 steps, one small file)
# Quick validation: trains 50 steps on within_us_ai_mythos_5k.jsonl (5K examples)
%cd /content/trimind-bitnet

!python training/train.py \
    --test_run \
    --train_file within_us_ai_mythos_5k.jsonl \
    --output_dir ./trimind-v1-test \
    --data_dir ./data \
    --batch_size 2 \
    --gradient_accumulation_steps 8 \
    --max_length 2048 \
    --lora_r 16 \
    --lora_alpha 32 \
    --logging_steps 5

In [ ]:
# @title 6. Train one file
%cd /content/trimind-bitnet

# Pick ONE file to train this session:
#   within_us_ai_mythos_5k.jsonl       (~5K)
#   within_us_ai_mythos_25k.jsonl      (~25K)
#   armand0e_fable_5.jsonl
#   victor_fable_worldcup.jsonl
#   roman_sonnet46.jsonl
#   roman_gemini31_code.jsonl
#   within_us_ai_opus48_5k.jsonl       (~5K)
#   norquinal_evol_210k.jsonl          (~210K - may need 2 sessions)
#   norquinal_evol_250k.jsonl          (~250K - may need 2 sessions)

TRAIN_FILE = "within_us_ai_mythos_5k.jsonl"  # smallest
OUTPUT_DIR = f"./trimind-v1-{TRAIN_FILE.replace('.jsonl','')}"

!python training/train.py \
    --train_file "{TRAIN_FILE}" \
    --output_dir "{OUTPUT_DIR}" \
    --data_dir ./data \
    --batch_size 2 \
    --gradient_accumulation_steps 8 \
    --max_length 4096 \
    --learning_rate 2e-4 \
    --lora_r 16 \
    --lora_alpha 32 \
    --num_epochs 1 \
    --save_steps 200 \
    --eval_steps 200 \
    --logging_steps 10 \
    --warmup_ratio 0.03 \
    --use_wandb true \
    --wandb_project trimind-bitnet \
    --push_to_hub true \
    --hub_model_id Abdulhadi446/Trimind-v1

In [ ]:
# @title 7. Resume interrupted training (same file)
%cd /content/trimind-bitnet

# If session disconnected mid-file, rerun with auto-resume:
TRAIN_FILE = "within_us_ai_mythos_5k.jsonl"
OUTPUT_DIR = f"./trimind-v1-{TRAIN_FILE.replace('.jsonl','')}"

!python training/train.py \
    --train_file "{TRAIN_FILE}" \
    --output_dir "{OUTPUT_DIR}" \
    --data_dir ./data \
    --batch_size 2 \
    --gradient_accumulation_steps 8 \
    --max_length 4096 \
    --learning_rate 2e-4 \
    --lora_r 16 \
    --lora_alpha 32 \
    --num_epochs 1 \
    --save_steps 200 \
    --eval_steps 200 \
    --logging_steps 10 \
    --warmup_ratio 0.03 \
    --use_wandb true \
    --wandb_project trimind-bitnet \
    --push_to_hub true \
    --hub_model_id Abdulhadi446/Trimind-v1

In [ ]:
# @title 8. Save data cache to Google Drive
# After first run, cache the dataset so you don't redownload each session
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
src = "/content/trimind-bitnet/data/raw"
dst = "/content/drive/MyDrive/trimind_data/raw"
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Copied {src} to {dst}")
else:
    print("Already cached or nothing to copy.")

In [ ]:
# @title 9. Push all adapters to Hub
from huggingface_hub import HfApi, create_repo

API = HfApi()
REPO = "Abdulhadi446/Trimind-v1"
create_repo(REPO, exist_ok=True, private=True)

adapters_dir = "/content/trimind-bitnet/trimind-v1-final/adapters"
if os.path.exists(adapters_dir):
    for f in os.listdir(adapters_dir):
        sub = os.path.join(adapters_dir, f)
        if os.path.isdir(sub):
            API.upload_folder(
                folder_path=sub,
                repo_id=REPO,
                repo_type="model",
                path_in_repo=f"adapters/{f}",
                commit_message=f"feat: LoRA adapters for {f}",
            )
            print(f"Uploaded {sub} to {REPO}/adapters/{f}")
print(f"\nHub repo: https://huggingface.co/{REPO}")